# Output Analysis

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import t2fpharm_study

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

In [2]:
def group_mean(group_cols: list[str]) -> pd.DataFrame:
    """Calculate mean of all numeric columns in the summary DataFrame, grouped by the specified columns.

    Parameters
    ----------
    group_cols
        List of column names to group by.

    Returns
    -------
    A new DataFrame with the group columns and the mean of each numeric column.
    """
    # Ensure grouping columns exist
    missing = set(group_cols) - set(summ.columns)
    if missing:
        raise ValueError(f"Grouping columns not in DataFrame: {missing}")

    # Identify numeric columns excluding the grouping columns
    numeric_cols = [col for col in summ.columns if col not in ["job_name", "job_idx", "group_id", "pdb_id"]]

    # Perform the groupby and mean aggregation
    result = summ.groupby(group_cols)[numeric_cols].mean().reset_index()
    return result

In [3]:
def scatter_plot(
    x_axis_col: str,
    y_axis_col: str,
    df: pd.DataFrame | None = None,
    ncols: int = 3
) -> None:
    """Plot x_axis_col vs. y_axis_col values for each job_name in the summary DataFrame.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with 't_all-n', 't_all-dp_lt2-l_all', and 'job_name' columns.
    jitter : float, optional
        Amount of uniform random jitter to add to x and y values to separate overlapping points.
    ncols : int, optional
        Number of columns in the subplot grid.

    Returns
    -------
    None
    """
    df = df if df is not None else sum_per_job
    data = df.copy()
    categories = data["job_name"].astype("category").cat.categories
    n = len(categories)
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(8 * ncols, 6 * nrows),
        sharex=True,
        sharey=True
    )
    axes = axes.flatten()

    for ax, job in zip(axes, categories):
        subset = data[data["job_name"] == job]
        x = subset[x_axis_col]
        y = subset[y_axis_col]
        ax.scatter(x, y, s=15, alpha=0.4, edgecolors='k', linewidths=0.2)
        ax.set_title(job)
        ax.set_xlabel(x_axis_col)
        ax.set_ylabel(y_axis_col)

    # Hide any unused subplots
    for ax in axes[n:]:
        ax.set_visible(False)

    fig.tight_layout()
    plt.show()

In [4]:
manager = t2fpharm_study.manager()

## Job Summary
Summaries of each job performed on each structure is available as a pandas DataFrame.
The dataFrame contains the following columns:
- `job_name`: Name of the job group.
- `job_idx`: Index of the job within its group. Each (`job_name`, `job_idx`) combination corresponds to a unique set of job input parameters.
- `group_id`: Group ID of the protein structure.
- `pdb_id`: PDB ID of the protein structure. Each row contains a unique (`job_name`, `job_idx`, `pdb_id`) combination and corresponds to a unique job.

The rest of the columns have the pattern `{feature type}-{data type}-{ligand type}`, as explained below.

### Feature Types
The first part of each column name indicates 
the specific feature type that the column's data correspond to.
Available values are:
- `t_all`: All types combined.
- `t_HD`: Hydrogen bond donor (hydrogen atom).
- `t_OA`: Hydrogen bond acceptor (oxygen atom).
- `t_C`: Hydrophobic (carbon atom).
- `t_e+`: Cationic (electrostatic field).
- `t_e-`: Anionic (electrostatic field).

### Data Types
The second part of each column name indicates
the type of data in that column.
These can be categorized into the following groups:

#### Match Distances
These correspond to distances between a matching pair of features
(i.e., one in the perceived target pharmacophore and a matching one in the derived ligand pharmacophore):
- `d_max`: Maximum distance.
- `d_mean`: Mean distance.
- `d_median`: Median distance.
- `d_min`: Minimum distance.

#### Match Ratios
These correspond to ratios (i.e. in range [0, 1]) with respect to the number of ligand features. 
- `dp_inf`: Ratio of unmatched features.
- `dp_lt1`: Ratio of matched features with a distance below 1 Å.
- `dp_lt2`: Ratio of matched features with a distance below 2 Å.
- `dp_lt3`: Ratio of matched features with a distance below 3 Å.

#### Counts
- `n`: Number of perceived target features.
- `nl_all`: Number of ligand features for all ligands in the structure's group.
- `nl_self`: Number of ligand features for the structure's own ligand.

#### Feature Radii
These are only available for `cnn` jobs
and contain data about the radii of perceived features,
i.e. the cluster radii.
- `r_max`: Maximum radius.
- `r_mean`: Mean radius.
- `r_min`: Minimum radius.

#### Feature Energy Values
These correspond to the field value at the center of perceived features.
- `v_max`: Maximum value.
- `v_mean`: Mean value.
- `v_min`: Minimum value.

### Ligand Types
The third part of the column name is only present for match columns
and indicates the ligand type the matching data corresponds to.
Available values are:
- `l_all`: All ligands in the structure's group.
- `l_self`: Only the ligand in the structure.

In [5]:
summ = manager.job_summary()
summ

,job_name,job_idx,group_id,pdb_id,t_all-d_max-l_all,t_all-d_max-l_self,t_all-d_mean-l_all,t_all-d_mean-l_self,t_all-d_median-l_all,t_all-d_median-l_self,t_all-d_min-l_all,t_all-d_min-l_self,t_all-dp_inf-l_all,t_all-dp_inf-l_self,t_all-dp_lt1-l_all,t_all-dp_lt1-l_self,t_all-dp_lt2-l_all,t_all-dp_lt2-l_self,t_all-dp_lt3-l_all,t_all-dp_lt3-l_self,t_all-n,t_all-nl_all,t_all-nl_self,t_all-r_max,t_all-r_mean,t_all-r_min,t_all-v_max,t_all-v_mean,t_all-v_min,t_C-d_max-l_all,t_C-d_max-l_self,t_C-d_mean-l_all,t_C-d_mean-l_self,t_C-d_median-l_all,t_C-d_median-l_self,t_C-d_min-l_all,t_C-d_min-l_self,t_C-dp_inf-l_all,t_C-dp_inf-l_self,t_C-dp_lt1-l_all,t_C-dp_lt1-l_self,t_C-dp_lt2-l_all,t_C-dp_lt2-l_self,t_C-dp_lt3-l_all,t_C-dp_lt3-l_self,t_C-n,t_C-nl_all,t_C-nl_self,t_C-r_max,t_C-r_mean,t_C-r_min,t_C-v_max,t_C-v_mean,t_C-v_min,t_HD-d_max-l_all,t_HD-d_max-l_self,t_HD-d_mean-l_all,t_HD-d_mean-l_self,t_HD-d_median-l_all,t_HD-d_median-l_self,t_HD-d_min-l_all,t_HD-d_min-l_self,t_HD-dp_inf-l_all,t_HD-dp_inf-l_self,t_HD-dp_lt1-l_all,t_HD-dp_lt1-l_self,t_HD-dp_lt2-l_all,t_HD-dp_lt2-l_self,t_HD-dp_lt3-l_all,t_HD-dp_lt3-l_self,t_HD-n,t_HD-nl_all,t_HD-nl_self,t_HD-r_max,t_HD-r_mean,t_HD-r_min,t_HD-v_max,t_HD-v_mean,t_HD-v_min,t_OA-d_max-l_all,t_OA-d_max-l_self,t_OA-d_mean-l_all,t_OA-d_mean-l_self,t_OA-d_median-l_all,t_OA-d_median-l_self,t_OA-d_min-l_all,t_OA-d_min-l_self,t_OA-dp_inf-l_all,t_OA-dp_inf-l_self,t_OA-dp_lt1-l_all,t_OA-dp_lt1-l_self,t_OA-dp_lt2-l_all,t_OA-dp_lt2-l_self,t_OA-dp_lt3-l_all,t_OA-dp_lt3-l_self,t_OA-n,t_OA-nl_all,t_OA-nl_self,t_OA-r_max,t_OA-r_mean,t_OA-r_min,t_OA-v_max,t_OA-v_mean,t_OA-v_min,t_e+-d_max-l_all,t_e+-d_max-l_self,t_e+-d_mean-l_all,t_e+-d_mean-l_self,t_e+-d_median-l_all,t_e+-d_median-l_self,t_e+-d_min-l_all,t_e+-d_min-l_self,t_e+-dp_inf-l_all,t_e+-dp_inf-l_self,t_e+-dp_lt1-l_all,t_e+-dp_lt1-l_self,t_e+-dp_lt2-l_all,t_e+-dp_lt2-l_self,t_e+-dp_lt3-l_all,t_e+-dp_lt3-l_self,t_e+-n,t_e+-nl_all,t_e+-nl_self,t_e+-r_max,t_e+-r_mean,t_e+-r_min,t_e+-v_max,t_e+-v_mean,t_e+-v_min,t_e--d_max-l_all,t_e--d_max-l_self,t_e--d_mean-l_all,t_e--d_mean-l_self,t_e--d_median-l_all,t_e--d_median-l_self,t_e--d_min-l_all,t_e--d_min-l_self,t_e--dp_inf-l_all,t_e--dp_inf-l_self,t_e--dp_lt1-l_all,t_e--dp_lt1-l_self,t_e--dp_lt2-l_all,t_e--dp_lt2-l_self,t_e--dp_lt3-l_all,t_e--dp_lt3-l_self,t_e--n,t_e--nl_all,t_e--nl_self,t_e--r_max,t_e--r_mean,t_e--r_min,t_e--v_max,t_e--v_mean,t_e--v_min
0,grid_search-cnn-1,0,A2AR,2YDO,10.923259,10.923259,7.864268,8.152646,7.294803,7.57666,4.773412,6.534006,0.3125,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3,16,4,1.0034,0.751021,0.374565,-0.641,-1.030667,-1.257,10.923259,<NA>,7.864268,<NA>,7.294803,<NA>,4.773412,<NA>,0.3125,<NA>,0.0,<NA>,0.0,<NA>,0.0,<NA>,0,16,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,10.923259,10.923259,7.864268,8.152646,7.294803,7.57666,4.773412,6.534006,0.3125,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,16,4,0.374565,0.374565,0.374565,-0.641,-0.641,-0.641,10.923259,10.923259,7.864268,8.152646,7.294803,7.57666,4.773412,6.534006,0.3125,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2,16,4,1.0034,0.939248,0.875097,-1.194,-1.2255,-1.257,10.923259,<NA>,7.864268,<NA>,7.294803,<NA>,4.773412,<NA>,0.3125,<NA>,0.0,<NA>,0.0,<NA>,0.0,<NA>,0,16,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,0,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,grid_search-cnn-1,0,A2AR,2YDV,12.378593,12.378593,5.762719,6.67163,3.487594,3.355216,2.67782,2.67782,0.3125,0.0,0.0,0.0,0.0,0.0,0.125,0.4,5,16,5,1.092109,0.788588,0.611283,-0.654,-1.0858,-1.34,12.378593,<NA>,5.762719,<NA>,3.487594,<NA>,2.67782,<NA>,0.3125,<NA>,0.0,<NA>,0.0,<NA>,0.125,<NA>,0,16,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,12.378593,12.378593,5.762719,6.67163,3.487594,3.355216,2.67782,2.67782,0.3125,0.0,0.0,0.0,0.0,0.0,0.125,0.4,1,16,5,0.611283,0.611283,0.611283,-0.654,-0.654,-0.654,12.378593,12.378593,5.762719,6.67163,3.487594,3.355216,2.67782,2.67782,0.3125,0.0,0.0,0.0,0.0,0.0,0.125,0.4,4,16,5,1.092109,0.832914,0.648755,-1.096,-1.19375,-1.34,12.378593,<NA>,5.762719,<NA>,3.

In [ ]:
# Execute if interested in ref structures only

# ref_pdb_ids = manager.dataset[manager.dataset["is_ref"]]["pdb_id"]
# ref_mask = summ["pdb_id"].isin(ref_pdb_ids)
# summ = summ[ref_mask]

In [6]:
char_dist = 2
for feature_type in ("all", "HD", "C", "OA", "e+", "e-"):
    col_type = f"t_{feature_type}"
    # In order for NaN values calculated by NumPy to be recognized by pandas,
    # we have to cast the input columns to NumPy first.
    # see: https://github.com/pandas-dev/pandas/issues/61758
    dp_lt2 = summ[f"{col_type}-dp_lt2-l_all"].astype(float)
    n_l = summ[f"{col_type}-nl_all"].astype(int)
    n_r = summ[f"{col_type}-n"].astype(int)
    d_mean = summ[f"{col_type}-d_mean-l_all"].astype(float)
    
    precision = dp_lt2 * n_l / n_r
    sensitivity = dp_lt2
    f1_score = (2 * precision * sensitivity / (precision + sensitivity)).fillna(0)
    weights = np.maximum(0, 1 - d_mean / char_dist)
    f1_score_weighted = f1_score * weights
    
    summ[f"{col_type}-simple_score"] = dp_lt2 / n_r
    summ[f"{col_type}-precision"] = precision 
    summ[f"{col_type}-sensitivity"] = sensitivity
    summ[f"{col_type}-f1_score"] = f1_score
    summ[f"{col_type}-f1_score_weighted"] = f1_score_weighted

In [7]:
sum_per_job = group_mean(["job_name", "job_idx"])

In [ ]:
scatter_plot("t_all-n", "t_all-dp_lt2-l_all")

In [ ]:
scatter_plot("t_HD-n", "t_HD-dp_lt2-l_all")

In [ ]:
scatter_plot("t_OA-n", "t_OA-dp_lt2-l_all")

In [ ]:
scatter_plot("t_C-n", "t_C-dp_lt2-l_all")

In [ ]:
scatter_plot("t_e+-n", "t_e+-dp_lt2-l_all")

In [ ]:
scatter_plot("t_e--n", "t_e--dp_lt2-l_all")

In [ ]:
scatter_plot("t_all-n", "t_all-f1_score_weighted")

In [21]:
x = sum_per_job.sort_values("t_all-f1_score_weighted", ascending=False)
x[(x["t_all-n"] < 60) & (x["job_name"] == "grid_search-cnn-1")][:10]
# x[:100]

,job_name,job_idx,t_all-d_max-l_all,t_all-d_max-l_self,t_all-d_mean-l_all,t_all-d_mean-l_self,t_all-d_median-l_all,t_all-d_median-l_self,t_all-d_min-l_all,t_all-d_min-l_self,t_all-dp_inf-l_all,t_all-dp_inf-l_self,t_all-dp_lt1-l_all,t_all-dp_lt1-l_self,t_all-dp_lt2-l_all,t_all-dp_lt2-l_self,t_all-dp_lt3-l_all,t_all-dp_lt3-l_self,t_all-n,t_all-nl_all,t_all-nl_self,t_all-r_max,t_all-r_mean,t_all-r_min,t_all-v_max,t_all-v_mean,t_all-v_min,t_C-d_max-l_all,t_C-d_max-l_self,t_C-d_mean-l_all,t_C-d_mean-l_self,t_C-d_median-l_all,t_C-d_median-l_self,t_C-d_min-l_all,t_C-d_min-l_self,t_C-dp_inf-l_all,t_C-dp_inf-l_self,t_C-dp_lt1-l_all,t_C-dp_lt1-l_self,t_C-dp_lt2-l_all,t_C-dp_lt2-l_self,t_C-dp_lt3-l_all,t_C-dp_lt3-l_self,t_C-n,t_C-nl_all,t_C-nl_self,t_C-r_max,t_C-r_mean,t_C-r_min,t_C-v_max,t_C-v_mean,t_C-v_min,t_HD-d_max-l_all,t_HD-d_max-l_self,t_HD-d_mean-l_all,t_HD-d_mean-l_self,t_HD-d_median-l_all,t_HD-d_median-l_self,t_HD-d_min-l_all,t_HD-d_min-l_self,t_HD-dp_inf-l_all,t_HD-dp_inf-l_self,t_HD-dp_lt1-l_all,t_HD-dp_lt1-l_self,t_HD-dp_lt2-l_all,t_HD-dp_lt2-l_self,t_HD-dp_lt3-l_all,t_HD-dp_lt3-l_self,t_HD-n,t_HD-nl_all,t_HD-nl_self,t_HD-r_max,t_HD-r_mean,t_HD-r_min,t_HD-v_max,t_HD-v_mean,t_HD-v_min,t_OA-d_max-l_all,t_OA-d_max-l_self,t_OA-d_mean-l_all,t_OA-d_mean-l_self,t_OA-d_median-l_all,t_OA-d_median-l_self,t_OA-d_min-l_all,t_OA-d_min-l_self,t_OA-dp_inf-l_all,t_OA-dp_inf-l_self,t_OA-dp_lt1-l_all,t_OA-dp_lt1-l_self,t_OA-dp_lt2-l_all,t_OA-dp_lt2-l_self,t_OA-dp_lt3-l_all,t_OA-dp_lt3-l_self,t_OA-n,t_OA-nl_all,t_OA-nl_self,t_OA-r_max,t_OA-r_mean,t_OA-r_min,t_OA-v_max,t_OA-v_mean,t_OA-v_min,t_e+-d_max-l_all,t_e+-d_max-l_self,t_e+-d_mean-l_all,t_e+-d_mean-l_self,t_e+-d_median-l_all,t_e+-d_median-l_self,t_e+-d_min-l_all,t_e+-d_min-l_self,t_e+-dp_inf-l_all,t_e+-dp_inf-l_self,t_e+-dp_lt1-l_all,t_e+-dp_lt1-l_self,t_e+-dp_lt2-l_all,t_e+-dp_lt2-l_self,t_e+-dp_lt3-l_all,t_e+-dp_lt3-l_self,t_e+-n,t_e+-nl_all,t_e+-nl_self,t_e+-r_max,t_e+-r_mean,t_e+-r_min,t_e+-v_max,t_e+-v_mean,t_e+-v_min,t_e--d_max-l_all,t_e--d_max-l_self,t_e--d_mean-l_all,t_e--d_mean-l_self,t_e--d_median-l_all,t_e--d_median-l_self,t_e--d_min-l_all,t_e--d_min-l_self,t_e--dp_inf-l_all,t_e--dp_inf-l_self,t_e--dp_lt1-l_all,t_e--dp_lt1-l_self,t_e--dp_lt2-l_all,t_e--dp_lt2-l_self,t_e--dp_lt3-l_all,t_e--dp_lt3-l_self,t_e--n,t_e--nl_all,t_e--nl_self,t_e--r_max,t_e--r_mean,t_e--r_min,t_e--v_max,t_e--v_mean,t_e--v_min,t_all-simple_score,t_all-precision,t_all-sensitivity,t_all-f1_score,t_all-f1_score_weighted,t_HD-simple_score,t_HD-precision,t_HD-sensitivity,t_HD-f1_score,t_HD-f1_score_weighted,t_C-simple_score,t_C-precision,t_C-sensitivity,t_C-f1_score,t_C-f1_score_weighted,t_OA-simple_score,t_OA-precision,t_OA-sensitivity,t_OA-f1_score,t_OA-f1_score_weighted,t_e+-simple_score,t_e+-precision,t_e+-sensitivity,t_e+-f1_score,t_e+-f1_score_weighted,t_e--simple_score,t_e--precision,t_e--sensitivity,t_e--f1_score,t_e--f1_score_weighted
1978,grid_search-cnn-1,1978,8.83559,6.162698,3.32634,3.014569,2.971279,2.654219,0.464616,0.930967,0.127647,0.132298,0.101784,0.123351,0.314345,0.365775,0.477959,0.519349,26.90625,61.875,8.75,4.57789,1.975175,0.887754,-0.309437,-0.689889,-1.168594,8.83559,6.355233,3.32634,3.131495,2.971279,2.783138,0.464616,0.957662,0.127647,0.145984,0.101784,0.115997,0.314345,0.328326,0.477959,0.490891,1.46875,61.875,8.28125,2.831363,2.32146,1.820976,-0.476208,-0.494139,-0.514167,8.83559,6.349638,3.32634,3.065102,2.971279,2.708167,0.464616,0.906671,0.127647,0.141118,0.101784,0.123241,0.314345,0.36516,0.477959,0.500639,9.46875,61.875,8.46875,2.606091,1.70244,1.025647,-0.516281,-0.581244,-0.639469,8.83559,6.301969,3.32634,2.84556,2.971279,2.469261,0.464616,0.709173,0.127647,0.162829,0.101784,0.151817,0.314345,0.427108,0.477959,0.544143,11.96875,61.875,7.71875,3.098926,1.754212,0.982301,-0.622469,-0.836442,-1.1565,10.044035,8.538306,3.752779,3.670567,3.269622,3.333882,0.456896,0.679229,0.088809,0.093537,0.087807,0.112411,0.302784,0.308941,0.484542,0.527746,2.59375,31.875,2.59375,3.454576

In [18]:
y = sum_per_job.sort_values("t_all-d_mean-l_all")
y[y["t_all-n"] < 60][:100]

,job_name,job_idx,t_all-d_max-l_all,t_all-d_max-l_self,t_all-d_mean-l_all,t_all-d_mean-l_self,t_all-d_median-l_all,t_all-d_median-l_self,t_all-d_min-l_all,t_all-d_min-l_self,t_all-dp_inf-l_all,t_all-dp_inf-l_self,t_all-dp_lt1-l_all,t_all-dp_lt1-l_self,t_all-dp_lt2-l_all,t_all-dp_lt2-l_self,t_all-dp_lt3-l_all,t_all-dp_lt3-l_self,t_all-n,t_all-nl_all,t_all-nl_self,t_all-r_max,t_all-r_mean,t_all-r_min,t_all-v_max,t_all-v_mean,t_all-v_min,t_C-d_max-l_all,t_C-d_max-l_self,t_C-d_mean-l_all,t_C-d_mean-l_self,t_C-d_median-l_all,t_C-d_median-l_self,t_C-d_min-l_all,t_C-d_min-l_self,t_C-dp_inf-l_all,t_C-dp_inf-l_self,t_C-dp_lt1-l_all,t_C-dp_lt1-l_self,t_C-dp_lt2-l_all,t_C-dp_lt2-l_self,t_C-dp_lt3-l_all,t_C-dp_lt3-l_self,t_C-n,t_C-nl_all,t_C-nl_self,t_C-r_max,t_C-r_mean,t_C-r_min,t_C-v_max,t_C-v_mean,t_C-v_min,t_HD-d_max-l_all,t_HD-d_max-l_self,t_HD-d_mean-l_all,t_HD-d_mean-l_self,t_HD-d_median-l_all,t_HD-d_median-l_self,t_HD-d_min-l_all,t_HD-d_min-l_self,t_HD-dp_inf-l_all,t_HD-dp_inf-l_self,t_HD-dp_lt1-l_all,t_HD-dp_lt1-l_self,t_HD-dp_lt2-l_all,t_HD-dp_lt2-l_self,t_HD-dp_lt3-l_all,t_HD-dp_lt3-l_self,t_HD-n,t_HD-nl_all,t_HD-nl_self,t_HD-r_max,t_HD-r_mean,t_HD-r_min,t_HD-v_max,t_HD-v_mean,t_HD-v_min,t_OA-d_max-l_all,t_OA-d_max-l_self,t_OA-d_mean-l_all,t_OA-d_mean-l_self,t_OA-d_median-l_all,t_OA-d_median-l_self,t_OA-d_min-l_all,t_OA-d_min-l_self,t_OA-dp_inf-l_all,t_OA-dp_inf-l_self,t_OA-dp_lt1-l_all,t_OA-dp_lt1-l_self,t_OA-dp_lt2-l_all,t_OA-dp_lt2-l_self,t_OA-dp_lt3-l_all,t_OA-dp_lt3-l_self,t_OA-n,t_OA-nl_all,t_OA-nl_self,t_OA-r_max,t_OA-r_mean,t_OA-r_min,t_OA-v_max,t_OA-v_mean,t_OA-v_min,t_e+-d_max-l_all,t_e+-d_max-l_self,t_e+-d_mean-l_all,t_e+-d_mean-l_self,t_e+-d_median-l_all,t_e+-d_median-l_self,t_e+-d_min-l_all,t_e+-d_min-l_self,t_e+-dp_inf-l_all,t_e+-dp_inf-l_self,t_e+-dp_lt1-l_all,t_e+-dp_lt1-l_self,t_e+-dp_lt2-l_all,t_e+-dp_lt2-l_self,t_e+-dp_lt3-l_all,t_e+-dp_lt3-l_self,t_e+-n,t_e+-nl_all,t_e+-nl_self,t_e+-r_max,t_e+-r_mean,t_e+-r_min,t_e+-v_max,t_e+-v_mean,t_e+-v_min,t_e--d_max-l_all,t_e--d_max-l_self,t_e--d_mean-l_all,t_e--d_mean-l_self,t_e--d_median-l_all,t_e--d_median-l_self,t_e--d_min-l_all,t_e--d_min-l_self,t_e--dp_inf-l_all,t_e--dp_inf-l_self,t_e--dp_lt1-l_all,t_e--dp_lt1-l_self,t_e--dp_lt2-l_all,t_e--dp_lt2-l_self,t_e--dp_lt3-l_all,t_e--dp_lt3-l_self,t_e--n,t_e--nl_all,t_e--nl_self,t_e--r_max,t_e--r_mean,t_e--r_min,t_e--v_max,t_e--v_mean,t_e--v_min,t_all-simple_score,t_all-precision,t_all-sensitivity,t_all-f1_score,t_all-f1_score_weighted,t_HD-simple_score,t_HD-precision,t_HD-sensitivity,t_HD-f1_score,t_HD-f1_score_weighted,t_C-simple_score,t_C-precision,t_C-sensitivity,t_C-f1_score,t_C-f1_score_weighted,t_OA-simple_score,t_OA-precision,t_OA-sensitivity,t_OA-f1_score,t_OA-f1_score_weighted,t_e+-simple_score,t_e+-precision,t_e+-sensitivity,t_e+-f1_score,t_e+-f1_score_weighted,t_e--simple_score,t_e--precision,t_e--sensitivity,t_e--f1_score,t_e--f1_score_weighted
29287,grid_search-lp-1,2071,5.965079,3.330714,1.821246,1.529312,1.420314,1.286471,0.30873,0.520062,0.0,0.0,0.291598,0.388777,0.704738,0.787607,0.843817,0.891305,47.40625,61.875,8.75,<NA>,<NA>,<NA>,-0.365875,-0.631354,-1.380406,5.965079,3.303524,1.821246,1.485126,1.420314,1.233079,0.30873,0.490329,0.0,0.0,0.291598,0.396236,0.704738,0.807014,0.843817,0.907072,23.6875,61.875,8.28125,<NA>,<NA>,<NA>,-0.377531,-0.526054,-0.742469,5.965079,3.465005,1.821246,1.568049,1.420314,1.312858,0.30873,0.50876,0.0,0.0,0.291598,0.364695,0.704738,0.773447,0.843817,0.884059,6.0,61.875,8.46875,<NA>,<NA>,<NA>,-0.503937,-0.535208,-0.569312,5.965079,3.694761,1.821246,1.608479,1.420314,1.305038,0.30873,0.497949,0.0,0.0,0.291598,0.381791,0.704738,0.749582,0.843817,0.871716,11.71875,61.875,7.71875,<NA>,<NA>,<NA>,-0.483375,-0.713381,-0.993687,6.323047,4.807536,2.042421,2.013555,1.517513,1.490593,0.299438,0.483047,0.0,0.0,0.274139,0.298773,0.631577,0.651313,0.780749,0.785512,3.0,31.875,2.59375,<NA>,<NA>,<NA>,-0.952406,-1.114646,-1.334594,7.399697,5.139655,2.371046,2.127811,1.703837,1.544173,0.2174

In [11]:
jobs = manager.job_inputs()

In [15]:
jobs[(jobs["job_name"] == "grid_search-lp-1") & (jobs["job_idx"] == 454)]

,job_name,job_idx,best_per_point,center_type,filter_function,filter_gaussian_sigma,filter_gaussian_sigma_mult,filter_gaussian_sigma_mult_idx,filter_radius,filter_radius_mult,filter_radius_mult_idx,max_distance,max_distance_mult,max_distance_mult_idx,max_features,max_features_mult,max_features_mult_idx,max_members,max_members_factor,max_members_factor_idx,min_distance,min_distance_mult,min_distance_mult_idx,min_members,min_members_percent,min_members_percent_idx,min_neighbors,min_neighbors_list_length,min_neighbors_list_length_idx,min_neighbors_start_percent,min_neighbors_start_percent_idx,priority_factor,priority_factor_idx,threshold_percentile,threshold_percentile_idx,threshold_value,threshold_value_idx
27670,grid_search-lp-1,454,NaN,NaN,gaussian,"{'C': 0.2125, 'HD': 0.15, 'OA': 0.19375, 'e+':...",0.25,0.0,"{'C': 0.85, 'HD': 0.6, 'OA': 0.775, 'e+': 1.11...",0.5,0.0,NaN,NaN,NaN,"{'C': 160, 'HD': 40, 'OA': 80, 'e+': 20, 'e-':...",20.0,6.0,NaN,NaN,NaN,"{('C', 'C'): 1.125, ('HD', 'HD'): 1.7999999999...",0.75,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'C': 1, 'HD': 1, 'OA': 0.5, 'e+': 0.3, 'e-': ...",4.0,NaN,NaN,0,0


In [ ]:
sum_per_job.sort_values("t_HD-f1_score_weighted", ascending=False)[:10]

In [ ]:
sum_per_job.sort_values("t_OA-f1_score_weighted", ascending=False)[:10]

In [ ]:
sum_per_job.sort_values("t_C-f1_score_weighted", ascending=False)[:10]

In [ ]:
sum_per_job.sort_values("t_e+-f1_score_weighted", ascending=False)[:10]

In [ ]:
sum_per_job.sort_values("t_e--f1_score_weighted", ascending=False)[:10]

In [ ]:
sum_per_job[sum_per_job["t_all-n"] < 100].sort_values("t_all-dp_lt2-l_all", ascending=False)[:10]

In [ ]:
sum_per_job[sum_per_job["t_all-n"] < 50].sort_values("t_all-dp_lt2-l_all", ascending=False)[:10]

In [ ]:
sum_per_job[sum_per_job["t_all-n"] < 30].sort_values("t_all-dp_lt2-l_all", ascending=False)[:10]

In [ ]:
sum_per_job[sum_per_job["t_all-n"] < 20].sort_values("t_all-dp_lt2-l_all", ascending=False)[:10]